In [728]:
import pandas as pd
import numpy as np
import re
import notebook_setup
#importo sqlalchemy y mi variable de entorno para conectarme a la base de datos
from sqlalchemy import text
#from dotenv import load_dotenv
#import unicodedata
from scripts.connection import get_engine

In [729]:

engine=get_engine()

# Carga desde la capa bronce

Aquí realizaré una carga desde la capa bronce. Usaré SQLAlchemy para hacerlo.

In [730]:
#leo el archivo
df_pato=pd.read_sql("SELECT * FROM bronze.datosctes_consultas_patologia", engine)
#Cuando se carga desde la base de datos, ocurre que los NULL se convierten en '#N/A' para los strings
#Y se convierten en None para los campos enteros
#Para facilitar el trabajo, convertiremos los #N/A es NaN
df_pato=df_pato.replace("#N/A",np.nan)
df_pato = df_pato.replace({None: np.nan})
df_pato.info()

C:\Users\espin\AppData\Local\Temp\ipykernel_1520\1783731738.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_pato = df_pato.replace({None: np.nan})


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167201 entries, 0 to 167200
Data columns (total 32 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id_saps               66424 non-null   object 
 1   saps                  166147 non-null  object 
 2   fecha                 167200 non-null  object 
 3   mes                   167201 non-null  int64  
 4   anio                  167201 non-null  int64  
 5   patologia_desc        166916 non-null  object 
 6   agrupacion_cie10      166913 non-null  object 
 7   patologia_cod         167175 non-null  object 
 8   id_rango_etario       166135 non-null  object 
 9   id_sexo               166136 non-null  float64
 10  consulta_cantidad     166132 non-null  object 
 11  rango_etario          166129 non-null  object 
 12  sexo                  166053 non-null  object 
 13  barrio_del_operativo  4953 non-null    object 
 14  unnamed_14            0 non-null       float64
 15  

La primera transformación que se hará es eliminar las columnas que no aportan información

In [731]:
#Como hay columnas que no nos interesa, establezco las columnas pertinentes para el análisis
columnas_requeridas=['id_saps','saps','fecha','patologia_desc','agrupacion_cie10','patologia_cod','id_rango_etario','consulta_cantidad','rango_etario','sexo']
#sólo tomo las columnas que van a ser utilizadas
df_pato=df_pato[columnas_requeridas]
df_pato.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167201 entries, 0 to 167200
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   id_saps            66424 non-null   object
 1   saps               166147 non-null  object
 2   fecha              167200 non-null  object
 3   patologia_desc     166916 non-null  object
 4   agrupacion_cie10   166913 non-null  object
 5   patologia_cod      167175 non-null  object
 6   id_rango_etario    166135 non-null  object
 7   consulta_cantidad  166132 non-null  object
 8   rango_etario       166129 non-null  object
 9   sexo               166053 non-null  object
dtypes: object(10)
memory usage: 12.8+ MB


Empecemos adaptando los tipos de datos. Por ejemplo, consulta_cantidad debería ser entero, mientras que fecha debería ser de tipo date. Realizando un primer análisis de la consulta_cantidad, notamos los siguientes resultados

In [732]:
df_pato['consulta_cantidad'].unique()

array(['18', '20', '22', '25', '1', '2', '4', '5', '15', '10', '3', '7',
       '9', '6', '8', '13', '12', '14', '11', '16', '37', '21', '31',
       '48', '29', '28', '32', '24', '26', '19', '27', '30', '17', '51',
       '49', '57', '39', '55', '42', '33', '40', '45', '38', '58', '23',
       '43', '47', '41', '35', '66', '50', '44', '46', '99', '83', '34',
       '71', '60', '|1', '53', '61', '36', '59', '63', '62', '52', '86',
       '80', '70', '56', '68', '78', '4.', '0', '117', '76', '1.', '.30',
       '79', '98', '54', '74', '64', '.7', '69', '8+', '73', '.5', '82',
       '.1', '95', '.2', '136', '170', nan, '123', '142', '119', '3.',
       '85', '88', '114', '92', '65', '104', '84', '135', '106', '4+',
       '198', '75', '154', '8+4', '89', '96', '77', '141', '111', '67',
       '321', '72', '112', '0,6', '138', '122', '133', '81', '163', '8/',
       '5+', '314', '188'], dtype=object)

Vemos algunos errores como ser 
- "4+" o "|1". 

Para esos casos vamos a eliminar los símbolos que no corresponden. Por ejemplo, podemos establecer que "4+" es en realidad 4 y para "|1" es el realidad 1.

Luego existen casos como:
- ".30"
- "0.2" 

En este caso, suponemos que el punto y los valores a la izquierda del punto son errores de tipeo. Esto lo establecemos así porque la cantidad de consultas debe ser un número entero mayor o igual a cero. Sería poco probable que los profesionales de la salud colocaran valores decimales.

Para el caso de "8+4", suponemos que se intentó realizar la suma y no se terminó de expresar la operación, quedando en formato de string.
Los valores nan los transformamos a "-1" para indicar la ausencia de valores para ese día.

In [733]:
#limpio los datos para consulta_cantidad
#Acciones:
#Reemplazar comas por puntos para indicar los decimales
#Eliminar los puntos y números antes del punto
#Eliminar signos como +, |, /, etc
#Reemplazar los valores nan por -1
df_pato["consulta_cantidad"] = (
    df_pato["consulta_cantidad"]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.replace("8+4", "12", regex=False)
        .str.replace(".7", "7", regex=False)
        .str.replace("0.5", "5", regex=False) 
        .str.replace(".30", "30", regex=False) 
        .str.replace("0.1", "1", regex=False) 
        .str.replace("0.2", "2", regex=False) 
        .str.replace("0.6", "6", regex=False)  
        .str.replace("+", "", regex=False)         
        .str.replace("|", "", regex=False)
        .str.replace("/", "", regex=False)
        .str.replace("nan", "-1",regex=False)
        .str.replace("None","-1")
        .str.strip()
        .str.replace(".0","",regex=False)
        .str.replace(".","",regex=False)
        .astype(int)  
)

df_pato.isnull().sum()


id_saps              100777
saps                   1054
fecha                     1
patologia_desc          285
agrupacion_cie10        288
patologia_cod            26
id_rango_etario        1066
consulta_cantidad         0
rango_etario           1072
sexo                   1148
dtype: int64

# Analizando El Campo Fecha

Nuevamente, revisemos los valores únicos para las fechas

In [734]:
df_pato['fecha'].unique()

array(['2020-01-01', '01-01-2020', '01-02-2020', '01-03-2020',
       '01-04-2020', '01-05-2020', '01/05', '01-06-2020', '01-07-2020',
       '01-08-2020', '02-08-2020', '05-08-2020', '01-09-2020',
       '01-10-2020', '01-11-2020', '01-12-2020', '01-01-2021',
       '01/01/2021', '01-02-2021', '01-03-2021', '03-2021', '01-04-2021',
       '01-05-2021', '01-06-2021', '02-06-2021', '04-06-2021',
       '09-06-2021', '11-06-2021', '16-06-2021', '25-06-2021',
       '30-06-2021', '01-07-2021', '08-07-2021', '14-07-2021',
       '15-07-2021', '23-07-2021', '28-07-2021', '30-07-2021',
       '07-07-2021', '21-07-2021', '02-07-2021', '01-08-2021',
       '04-08-2021', '06-08-2021', '01-09-2021', '12-08-2021',
       '05-08-2021', '20-08-2021', '18-08-2021', '11-08-2021',
       '01-10-2021', '01-11-2021', '01/11/2021', '13-08-2021',
       '03-09-2021', '03-11-2021', '23-11-2021', '17-09-2021',
       '27-10-2021', '29-10-2021', '06-10-2021', '12-11-2021',
       '24-11-2021', '17-11-2021', 

Notamos algunos valores extraños como 

- "05-01" 
- "03-2021" 

En el primer caso no tenemos el año, en el segundo no tenemos el día. Luego, existen fechas donde el separador es "-" y en otras es "/". Tenemos también valores nan.
Por último, el formato para algunas fechas es día-mes-año.
Reemplazaremos los "/" por "-", colocaremos el formato año-mes-dia y cualquier campo que sea nan o que no se pueda convertir en fecha, le daremos el valor 1900-01-01 para indicar la ausencia de fecha. Eso lo haremos con la siguiente función:

In [735]:
def normalizar_fecha(fecha_entrada):
    '''
    Permite normalizar las fechas a un formato dd-mm-yyyy y establecer que un string que no
    tenga formato de fecha sea convertido a 1900-01-01
    
    Parámetros
    
    fecha_entrada: variable string que representa la fecha pasada desde el conjunto de datos
    
    Retorno
    
    Retorna una fecha en formato yyyy-mm-dd
    
    '''
    FECHA_NULA = pd.Timestamp("1900-01-01")
    # 1. NaN directo
    if pd.isna(fecha_entrada):
        return FECHA_NULA
    
    fecha_entrada= str(fecha_entrada).strip()
    
    #o formato dd-mm, mm-dd, dd-yyyy, mm-yyyy
    if re.fullmatch(r"\d{2}-\d{2}", fecha_entrada) or re.fullmatch(r"\d{2}-\d{4}", fecha_entrada) or re.fullmatch(r"\d{4}-\d{2}", fecha_entrada):
        return FECHA_NULA

    
    # 3. dd-mm-yyyy (con o sin hora)
    if re.match(r"^\d{2}-\d{2}-\d{4}", fecha_entrada):
        fecha = pd.to_datetime(
            fecha_entrada,
            format="%d-%m-%Y",
            errors="coerce"
        )
        return fecha.date() 

    # 4. yyyy-mm-dd (con o sin hora)
    if re.match(r"^\d{4}-\d{2}-\d{2}", fecha_entrada):
        fecha = pd.to_datetime(
            fecha_entrada,
            format="%Y-%m-%d",
            errors="coerce"
        )
        return fecha.date()

    # 5. Todo lo demás → fecha nula
    return FECHA_NULA

Cuando se defina la función, aplicaremos esa función a los campos fecha de los conjuntos de datos. Cuando el formato esté correcto, podemos adaptar el tipo de dato a fecha sin problemas, en caso de ser necesario, usando pd.to_datetime.

In [736]:
#transformamos a string para facilitar el trabajo, reemplazamos "/" por "-" y limpiamos
#los blancos
df_pato['fecha']=(
    df_pato['fecha']
    .astype(str)
    .str.replace("/","-")
    .str.strip()
)
print("El total de valores únicos para las fechas es {}".format(len(df_pato['fecha'].unique())))

El total de valores únicos para las fechas es 135


In [737]:
#aplico la función a cada fecha del dataframe
df_pato["fecha"] = df_pato["fecha"].apply(normalizar_fecha)
df_pato["fecha"].unique()

array([datetime.date(2020, 1, 1), datetime.date(2020, 2, 1),
       datetime.date(2020, 3, 1), datetime.date(2020, 4, 1),
       datetime.date(2020, 5, 1), Timestamp('1900-01-01 00:00:00'),
       datetime.date(2020, 6, 1), datetime.date(2020, 7, 1),
       datetime.date(2020, 8, 1), datetime.date(2020, 8, 2),
       datetime.date(2020, 8, 5), datetime.date(2020, 9, 1),
       datetime.date(2020, 10, 1), datetime.date(2020, 11, 1),
       datetime.date(2020, 12, 1), datetime.date(2021, 1, 1),
       datetime.date(2021, 2, 1), datetime.date(2021, 3, 1),
       datetime.date(2021, 4, 1), datetime.date(2021, 5, 1),
       datetime.date(2021, 6, 1), datetime.date(2021, 6, 2),
       datetime.date(2021, 6, 4), datetime.date(2021, 6, 9),
       datetime.date(2021, 6, 11), datetime.date(2021, 6, 16),
       datetime.date(2021, 6, 25), datetime.date(2021, 6, 30),
       datetime.date(2021, 7, 1), datetime.date(2021, 7, 8),
       datetime.date(2021, 7, 14), datetime.date(2021, 7, 15),
       d

In [738]:
#convierto finalmente en fecha
df_pato["fecha"]=pd.to_datetime(df_pato['fecha'])
print("El total de valores únicos para las fechas despues de la limpieza: {}".format(len(df_pato['fecha'].unique())))
df_pato["fecha"].unique()

El total de valores únicos para las fechas despues de la limpieza: 132


<DatetimeArray>
['2020-01-01 00:00:00', '2020-02-01 00:00:00', '2020-03-01 00:00:00',
 '2020-04-01 00:00:00', '2020-05-01 00:00:00', '1900-01-01 00:00:00',
 '2020-06-01 00:00:00', '2020-07-01 00:00:00', '2020-08-01 00:00:00',
 '2020-08-02 00:00:00',
 ...
 '2025-04-22 00:00:00', '2025-04-24 00:00:00', '2025-06-01 00:00:00',
 '2025-05-15 00:00:00', '2025-05-13 00:00:00', '2025-05-20 00:00:00',
 '2025-05-29 00:00:00', '2025-05-22 00:00:00', '2025-07-01 00:00:00',
 '2025-08-01 00:00:00']
Length: 132, dtype: datetime64[ns]

# Analizando Las Patologías

Empecemos analizando las situaciones en las que tanto el código de la patología como la agrupación cie10 tienen el valor NaN. Esto es importante porque nos dice qué registro no tienen ni el código ni su agrupación, lo cual nos hace muy difícil saber qué patología fue la que se trató en la consulta.

In [739]:
df_pato[['patologia_desc','agrupacion_cie10','patologia_cod']].head(10)

,patologia_desc,agrupacion_cie10,patologia_cod
0,CONTROL DEL NIÑO SANO - CONTROL DE SALUD,PERSONAS EN CONTACTO CON LOS SERVICIOS DE SALU...,Z001
1,CONTROL DEL NIÑO SANO - CONTROL DE SALUD,PERSONAS EN CONTACTO CON LOS SERVICIOS DE SALU...,Z001
2,CONTROL DEL NIÑO SANO - CONTROL DE SALUD,PERSONAS EN CONTACTO CON LOS SERVICIOS DE SALU...,Z001
3,CONTROL DEL NIÑO SANO - CONTROL DE SALUD,PERSONAS EN CONTACTO CON LOS SERVICIOS DE SALU...,Z001
4,CONTROL DEL NIÑO SANO - CONTROL DE SALUD,PERSONAS EN CONTACTO CON LOS SERVICIOS DE SALU...,Z001
5,CONTROL DEL NIÑO SANO - CONTROL DE SALUD,PERSONAS EN CONTACTO CON LOS SERVICIOS DE SALU...,Z001
6,DIARREA Y GASTROENTERITIS DE PRESUNTO ORIGEN I...,ENFERMEDADES INFECCIOSAS INTESTINALES,A09
7,DIARREA Y GASTROENTERITIS DE PRESUNTO ORIGEN I...,ENFERMEDADES INFECCIOSAS INTESTINALES,A09
8,DIARREA Y GASTROENTERITIS DE PRESUNTO ORIGEN I...,ENFERMEDADES INFECCIOSAS INTESTINALES,A09
9,DIARREA Y GASTROENTERITIS DE PRESUNTO ORIGEN I...,ENFERMEDADES INFECCIOSAS INTESTINALES,A09


In [740]:
df_pato[df_pato['patologia_cod']=='A071']

,id_saps,saps,fecha,patologia_desc,agrupacion_cie10,patologia_cod,id_rango_etario,consulta_cantidad,rango_etario,sexo
25979,73,SAN MARCOS,2020-09-01,NaN,NaN,A071,2,1,1 - 4 AÑOS,F


In [741]:
indices=df_pato[(df_pato['agrupacion_cie10'].isnull()) & (df_pato['patologia_cod'].isnull()) & (df_pato['patologia_desc'].isnull())].index
indices

Index([ 66743,  66794,  67028,  67029,  67032,  67034,  67035,  67036,  67040,
        72174,  75017,  75167,  75168,  75169,  75170,  75171,  75172,  75173,
        75174, 107598, 107599, 121592, 139698, 139699, 163654, 163655],
      dtype='int64')

Con los índices, sabemos en cuales registros se cumple la condición. Lo que queremos es convertir los nan en "n/a" para indica la ausencia de los datos. Con esto ya sabemos de cuales registros no podremos saber la patología en cuestión.

In [742]:
#modifico los valores nan a "n/a" para indicar la ausencia de valores para esos campos
df_pato.iloc[indices,3:6]="n/a"
df_pato.iloc[indices]



,id_saps,saps,fecha,patologia_desc,agrupacion_cie10,patologia_cod,id_rango_etario,consulta_cantidad,rango_etario,sexo
66743,42,Dr. SANTIAGO LORENZO,2021-11-01,n/a,n/a,n/a,5,31,20-39 AÑOS,F
66794,34,MARCELINO VERA,2021-11-01,n/a,n/a,n/a,6,1,40-69 AÑOS,F
67028,62,PUJOL,2021-11-01,n/a,n/a,n/a,3,3,5 - 14 AÑOS,F
67029,62,PUJOL,2021-11-01,n/a,n/a,n/a,3,3,5 - 14 AÑOS,M
67032,62,PUJOL,2021-11-01,n/a,n/a,n/a,6,1,40-69 AÑOS,F
67034,62,PUJOL,2021-11-01,n/a,n/a,n/a,2,1,1 - 4 AÑOS,M
67035,62,PUJOL,2021-11-01,n/a,n/a,n/a,3,1,5 - 14 AÑOS,F
67036,62,PUJOL,2021-11-01,n/a,n/a,n/a,3,1,5 - 14 AÑOS,M
67040,62,PUJOL,2021-11-01,n/a,n/a,n/a,4,1,15 - 19 AÑOS,F
72174,32,DR. MANUEL CASSUSO,2021-12-01,n/a,n/a,n/a,NaN,-1,NaN,NaN


Ahora, buscamos los registros donde tenemos el código cie10, pero no tenemos la agrupación. Conociendo el código, podremos conocer a qué agrupación pertenece.

In [743]:
indices=df_pato[(df_pato['agrupacion_cie10'].isnull()) & (df_pato['patologia_cod']).notnull()].index
df_pato.iloc[indices]

,id_saps,saps,fecha,patologia_desc,agrupacion_cie10,patologia_cod,id_rango_etario,consulta_cantidad,rango_etario,sexo
24480,38,DR. FLIER,2020-09-01,NaN,NaN,R78,3,2,5 - 14 AÑOS,M
24491,38,DR. FLIER,2020-09-01,NaN,NaN,U19,5,4,20-39 AÑOS,F
24798,NaN,DR. KORIMBLUM,2020-09-01,NaN,NaN,K104,5,1,20-39 AÑOS,F
24799,NaN,DR. KORIMBLUM,2020-09-01,NaN,NaN,K104,6,1,40-69 AÑOS,F
24906,75,DR. MAURICIO OPEN,2020-09-01,NaN,NaN,M910,5,1,20-39 AÑOS,F
...,...,...,...,...,...,...,...,...,...,...
161598,NaN,DR. PIRCHI,2025-06-01,NaN,NaN,Z71,5,9,20-39 AÑOS,F
161599,NaN,DR. PIRCHI,2025-06-01,NaN,NaN,Z71,5,1,20-39 AÑOS,M
161600,NaN,DR. PIRCHI,2025-06-01,NaN,NaN,Z71,6,1,40-69 AÑOS,F
163186,NaN,DR. KORIMBLUM,2025-07-01,NaN,NaN,G51,3,1,5 - 14 AÑOS,F


Notamos que tenemos el código de la patología, pero no la descripción y la agrupación. Podemos usar ciertas páginas web con esa información y, además, los datos abiertos de Corrientes también aporta un archivo con la información del código cie10. Las páginas utilizadas se encuentran en las referencias del README.
Cargaremos el csv con los códigos cie10 para buscar los que nos faltan.

In [744]:
#csv con los datos de las patologías y su código cie10
df_nomen=pd.read_sql("SELECT * FROM bronze.datosctes_cie10", engine)
df_nomen=df_nomen.replace("#N/A",np.nan)
df_nomen.head()

,id_patologia,tipo_patologia,descripcion
0,A00,ENFERMEDADES INFECCIOSAS INTESTINALES,COLERA
1,A01,ENFERMEDADES INFECCIOSAS INTESTINALES,FIEBRES TIFOIDEA Y PARATIFOIDEA
2,A02,ENFERMEDADES INFECCIOSAS INTESTINALES,OTRAS INFECCIONES DEBIDAS A SALMONELLA
3,A03,ENFERMEDADES INFECCIOSAS INTESTINALES,SHIGELOSIS
4,A04,ENFERMEDADES INFECCIOSAS INTESTINALES,OTRAS INFECCIONES INTESTINALES BACTERIANAS


El problema es que hay códigos cie10 que no figuran en el documento de referencia. Así que usaremos otras fuentes para conseguir la información. Esta fuente proviene de un buscador para las patología según su código cie10.
Para saber qué códigos cie10 no están presentes, compararemos qué códigos están presentes en el conjunto de datos de la patologías y los códigos presentes en el archivo con los datos de los códigos cie10. Con esta comparación sabremos qué códigos están ausentes en el archivo de códigos cie10. Estos códigos los buscaremos en páginas externas y, posteriormente, usaremos esos datos para limpiar los datos.

In [745]:
#obtenemos los códigos cie10 de la tablas de consultas y convierto la lista en un conjunto
set_cod_cie_pato=set(df_pato['patologia_cod'].unique())
#obtenemos los códigos cie10 del documento de referencia y convierto la lista en un conjunto
set_cod_cie=set(df_nomen['id_patologia'].unique())
#aplico la diferencia entre los códigos cie10 de las patologías y los códigos cie10 de la tabla de códigos
#el resultado tendrá los códigos cie10 que están en el conjunto de datos de patologías, pero no en la tabla de
#códigos cie10.
display(set_cod_cie_pato.difference(set_cod_cie))


{'A071',
 'B11',
 'B20',
 'B84',
 'C61',
 'E027',
 'E039',
 'E08',
 'E444',
 'F441',
 'F450',
 'G510',
 'H13',
 'H15',
 'H650',
 'H660',
 'H67',
 'I499',
 'I519',
 'I639',
 'I959',
 'J09',
 'J12',
 'J81',
 'K080',
 'K104',
 'K295',
 'K746',
 'K760',
 'K819',
 'L739',
 'M515',
 'M624',
 'M67',
 'M796',
 'M910',
 'N11',
 'N12',
 'N391',
 'N481',
 'N951',
 'R011',
 'R07',
 'R088',
 'R252',
 'R702',
 'R78',
 'S019',
 'S06',
 'S401',
 'S49',
 'S69',
 'S71',
 'S89',
 'S99',
 'T171',
 'T418',
 'T670',
 'T74',
 'T784',
 'U19',
 'U59',
 'U60',
 'U61',
 'U62',
 'U63',
 'U64',
 'U67',
 'V206',
 'Y20',
 'Y30',
 'Y45',
 'Z14',
 'Z300',
 'Z38',
 'a04',
 'a09',
 'a51',
 'a53',
 'a59',
 'b01',
 'b02',
 'b08',
 'b30',
 'b36',
 'b373',
 'b49',
 'b68',
 'b82',
 'b85',
 'b86',
 'd50',
 'e03',
 'e05',
 'e10',
 'e11',
 'e14',
 'e441',
 'e46',
 'e66',
 'e78',
 'f29',
 'f40',
 'f51',
 'f91',
 'g43',
 'g47',
 'h00',
 'h10',
 'h400',
 'h60',
 'h65',
 'h66',
 'h920',
 'i10',
 'i15',
 'i20',
 'i84',
 'i95',
 'j00

Lo que haré es crear un diccionario cuyas claves son los códigos cie10 y los valores contienen las descripciones de las patologías según ese código. Luego, emplearé ese diccionario para completar los valores faltantes. Este es un primer enfoque, ya que puede haber algunos códigos que se escapen. Y se realiza así porque son pocos valores, el trabajo de hacer esto es más rápido que ir reformateando los csv donde están los códigos cie10 obtenidos de otras fuentes. Posteriormente, se puede completar el diccionario con los código faltantes.
Aunque debe tenerse en cuenta que ese documento deberá reformatearse para futuras actualizaciones, ya que automatizaría mejor el proceso.

In [746]:
#el diccionario contiene los códigos cie10 y su agrupación que faltan en el conjunto de datos de las patologías.
#Los casos de códigos con 'n/a' son patologías cuyo código cie10 no encontramos.
map_cie_cod={
    'A071':'Giardiasis [lambliasis]',
    'B11':'n/a',
    "B82":"Parasitosis intestinal, sin otra especificación",
    'B20':'Enfermedad por virus de la inmunodeficiencia humana [VIH], resultante en enfermedades infecciosas y parasitarias',
    'B84':'n/a',
    'B084':'Estomatitis vesicular enteroviral con exantema',
    'C61':'Tumor maligno de la próstata',
    'E039':'Hipotiroidismo',
    "E027":"n/a",
    'E08':'n/a',
    'E444':'Desnutrición proteicocalórica de grado moderado y leve',
    'F441':'Fuga disociativa',
    'F450':'Trastorno de somatización',
    'G510':'Parálisis de Bell',
    "G51":'Trastornos del nervio facial',
    'H13':'Trastornos de la conjuntiva en enfermedades clasificadas en otra parte',
    'H15':'Trastornos de la esclerótica',
    'H650':'Otitis media aguda serosa',
    'H660':'Otitis media supurativa aguda',
    'H67':'Otitis media en enfermedades clasificadas en otra parte',
    'I499':'Arritmia cardíaca, no especificada',
    'I59':'n/a',
    'I519':'Enfermedad cardíaca, no especificada',
    'I639':'Infarto cerebral, no especificado',
    'I959':'Hipotensión, no especificada',
    'J09':'Influenza debida a virus de la influenza aviar identificado',
    'J12':'	Neumonía viral, no clasificada en otra parte',
    'J81':'Edema pulmonar',
    'K080':'Exfoliación de los dientes debida a causas sistémicas',
    'K104':'n/a',
    'K295':'Gastritis crónica, no especificada',
    'K746':'Otras cirrosis del hígado y las no especificadas',
    'K760':'Degeneración grasa del hígado, no clasificada en otra parte',
    'K819':'Colecistitis, no especificada',
    'L739':'Trastorno folicular, no especificado',
    'M515':'n/a',
    'M624':'Contractura muscular',
    'M67':'Otros trastornos de la sinovia y del tendón',
    'M796':'Dolor en miembro',
    'M910':'Osteocondrosis juvenil de la pelvis',
    'N11':'Nefritis tubulointersticial crónica',
    'N12':'Nefritis tubulointersticial, no especificada como aguda o crónica',
    'N391':'Proteinuria persistente, n o especificada',
    'N481':'Balanopostitis',
    'N951':'Estados menopáusicos y climatéricos femeninos',
    'R011':'Soplo cardíaco, no especificado',
    'R07':'Dolor de garganta y en el pecho',
    'R088':'n/a',
    'R252':'Calambres y espasmos',
    'R702':'n/a',
    'R78':'Hallazgo de drogas y otras sustancias que normalmente no se encuentran en la sangre',
    'S019':'Herida de la cabeza, parte no especificada',
    'S06':'Traumatismo intracraneal',
    "S401":'n/a',
    "S49":"Otros traumatismos y los no especificados del hombro y del brazo",
    "S69":"Otros traumatismos y los no especificados de la muñeca y de la mano",
    "S71":"Herida de la cadera y del muslo",
    "S89":"Otros traumatismos y los no especificados de la pierna",
    "S99":"Otros traumatismos y los no especificados del pie y del tobillo",
    "T171":"Cuerpo extraño en el orificio nasal",
    "T418":"n/a",
    "T670":"Golpe de calor e insolación",
    "T74":"Síndrome del maltrato",
    "T784":"Alergia no especificada",
    "U19":"n/a",
    "U59":"n/a",
    "U60":"n/a",
    "U61":"n/a",
    "U62":"n/a",
    "U63":"n/a",
    "U64":"n/a",
    "U67":"n/a",
    "V206":"n/a",
    "Y20":"Ahorcamiento, estrangulamiento y sofocación, de intención no determinada",
    "Y30":"Caída, salto o empujón desde lugar elevado, de intención no determinada",
    "Z14":"n/a",
    "Y40":"Efectos adversos de antibióticos sistémicos",
    "Y45":"Efectos adversos de drogas analgésicas, antipiréticas y antiinflamatorias",
    "Z300":"Consejo y asesoramiento general sobre la anticoncepción",
    "Z38":"Nacidos vivos según lugar de nacimiento",
    "Z71":"Personas en contacto con los servicios de salud por otras consultas y consejos médicos, no clasificados en otra parte",
    "Z73":"	Problemas relacionados con dificultades con el modo de vida",
    "nan":"n/a"

}

In [747]:
#recorro el índice de los datos donde existe el código cie10, pero no existe la explicación
#del código.
#En la columna donde debe ir la explicación de la patología según el código, coloco
#el valor del diccionario, ya que la clave es el código cie10
#{clave:valor}
#fila i y en la columna 4, que es el de la agrupación_cie10
for i in indices:
    df_pato.iloc[i,4]=map_cie_cod[df_pato.iloc[i,5]].strip()

In [748]:
df_pato.iloc[indices]

,id_saps,saps,fecha,patologia_desc,agrupacion_cie10,patologia_cod,id_rango_etario,consulta_cantidad,rango_etario,sexo
24480,38,DR. FLIER,2020-09-01,NaN,Hallazgo de drogas y otras sustancias que norm...,R78,3,2,5 - 14 AÑOS,M
24491,38,DR. FLIER,2020-09-01,NaN,n/a,U19,5,4,20-39 AÑOS,F
24798,NaN,DR. KORIMBLUM,2020-09-01,NaN,n/a,K104,5,1,20-39 AÑOS,F
24799,NaN,DR. KORIMBLUM,2020-09-01,NaN,n/a,K104,6,1,40-69 AÑOS,F
24906,75,DR. MAURICIO OPEN,2020-09-01,NaN,Osteocondrosis juvenil de la pelvis,M910,5,1,20-39 AÑOS,F
...,...,...,...,...,...,...,...,...,...,...
161598,NaN,DR. PIRCHI,2025-06-01,NaN,Personas en contacto con los servicios de salu...,Z71,5,9,20-39 AÑOS,F
161599,NaN,DR. PIRCHI,2025-06-01,NaN,Personas en contacto con los servicios de salu...,Z71,5,1,20-39 AÑOS,M
161600,NaN,DR. PIRCHI,2025-06-01,NaN,Personas en contacto con los servicios de salu...,Z71,6,1,40-69 AÑOS,F
163186,NaN,DR. KORIMBLUM,2025-07-01,NaN,Trastornos del nervio facial,G51,3,1,5 - 14 AÑOS,F


In [749]:
#luego, las descripciones de patologías que sean nulas, les coloco el valor 'n/a'
#estas descripciones no están relacionados directamente con el código cie10 y su agrupación, según las
#fuentes consultadas. Creemos que es un detalle propio que se le da
# en los saps, accesorio a la explicación de cie10. Así que los dejamos en 'n/a'
df_pato.iloc[df_pato[df_pato['patologia_desc'].isnull()].index,3]='n/a'
df_pato.isnull().sum()

id_saps              100777
saps                   1054
fecha                     0
patologia_desc            0
agrupacion_cie10          0
patologia_cod             0
id_rango_etario        1066
consulta_cantidad         0
rango_etario           1072
sexo                   1148
dtype: int64

# Trabajando con los rangos etarios

Primero, revisemos los valores únicos para los rangos etarios.

In [750]:
df_pato['rango_etario'].unique()

array(['< 1 AÑO', '1 - 4 AÑOS', '5 - 14 AÑOS', '15 - 19 AÑOS',
       '20-39 AÑOS', '40-69 AÑOS', '>=70 AÑOS', '#REF!', nan],
      dtype=object)

Busquemos los indices donde figuren "#REF!" y "nan". Esos valores los cambiaremos a "n/a"

In [751]:
index_etario=df_pato[(df_pato['rango_etario'].astype(str)=='nan') | (df_pato['rango_etario']=='#REF!')].index
df_pato.iloc[index_etario].head()


,id_saps,saps,fecha,patologia_desc,agrupacion_cie10,patologia_cod,id_rango_etario,consulta_cantidad,rango_etario,sexo
674,NaN,DR. ANIBAL MALVIDO,2020-01-01,ANEMIAS POR DEFICIENCIA DE HIERRO,ANEMIAS NUTRICIONALES,D50,1,2,#REF!,F
7231,71,VILLA CHIQUITA,2020-02-01,DERMATITIS ATOPICA,DERMATITIS Y ECZEMA,L20,3,1,#REF!,M
24685,48,DR. JOSE DE LOS REYES VIDAL,2020-09-01,GASTRITIS Y DUODENITIS,"ENFERMEDADES DEL ESOFAGO, DEL ESTOMAGO Y DEL D...",K29,||,1,NaN,M
32653,86,PASO MARTINEZ,2020-12-01,CONTROL DE SALUD: EXAMEN MEDICO GENERAL ADULTO,PERSONAS EN CONTACTO CON LOS SERVICIOS DE SALU...,Z000,4,7,#REF!,F
33356,NaN,DR. ANIBAL MALVIDO,2021-01-01,TRASTORNOS DE LOS TEJIDOS BLANDOS EN ENFERMEDA...,OTROS TRASTORNOS DE LOS TEJIDOS BLANDOS,M73,9,1,NaN,F


Ahora, queremos que la columna "rango_etario" tenga el valor 'n/a' para los índices encontrados. Eso lo hacemos de la siguiente forma

In [752]:
#Cambiamos los valores del rango etario con NaN en 'n/a'. A la vez, cambio también el campo id_rango_etario
#pertinente a 'n/a'. Esto es así porque un rango etario con 'n/a' no debería tener un id que lo identifique, a priori.
df_pato.iloc[index_etario,6:9:2]='n/a'
df_pato.iloc[index_etario]

,id_saps,saps,fecha,patologia_desc,agrupacion_cie10,patologia_cod,id_rango_etario,consulta_cantidad,rango_etario,sexo
674,NaN,DR. ANIBAL MALVIDO,2020-01-01,ANEMIAS POR DEFICIENCIA DE HIERRO,ANEMIAS NUTRICIONALES,D50,n/a,2,n/a,F
7231,71,VILLA CHIQUITA,2020-02-01,DERMATITIS ATOPICA,DERMATITIS Y ECZEMA,L20,n/a,1,n/a,M
24685,48,DR. JOSE DE LOS REYES VIDAL,2020-09-01,GASTRITIS Y DUODENITIS,"ENFERMEDADES DEL ESOFAGO, DEL ESTOMAGO Y DEL D...",K29,n/a,1,n/a,M
32653,86,PASO MARTINEZ,2020-12-01,CONTROL DE SALUD: EXAMEN MEDICO GENERAL ADULTO,PERSONAS EN CONTACTO CON LOS SERVICIOS DE SALU...,Z000,n/a,7,n/a,F
33356,NaN,DR. ANIBAL MALVIDO,2021-01-01,TRASTORNOS DE LOS TEJIDOS BLANDOS EN ENFERMEDA...,OTROS TRASTORNOS DE LOS TEJIDOS BLANDOS,M73,n/a,1,n/a,F
...,...,...,...,...,...,...,...,...,...,...
150744,NaN,DR. MAURICIO OPEN,2025-02-01,ARTRITIS REUMATOIDES SEROPOSITIVA,ARTROPATIAS INFECCIOSAS,M05,n/a,-1,n/a,NaN
155531,NaN,ESPERANZA,2025-04-01,DIABETES MELLITUS NO INSULINODEPENDIENTE,DIABETES MELLITUS,E11,n/a,-1,n/a,NaN
163654,NaN,SANTA MARTA,2025-07-01,n/a,n/a,n/a,n/a,-1,n/a,NaN
163655,NaN,SANTA MARTA,2025-07-01,n/a,n/a,n/a,n/a,-1,n/a,NaN


In [753]:
#reviso la situación de los nulos hasta el momento
df_pato.isnull().sum()

id_saps              100777
saps                   1054
fecha                     0
patologia_desc            0
agrupacion_cie10          0
patologia_cod             0
id_rango_etario           0
consulta_cantidad         0
rango_etario              0
sexo                   1148
dtype: int64

# Analizando el id para el rango etario

Empecemos viendo los valores que posee

In [754]:

display(df_pato['rango_etario'].unique())
display(df_pato['id_rango_etario'].unique())
display(df_pato[['rango_etario','id_rango_etario']].drop_duplicates().head(10))

array(['< 1 AÑO', '1 - 4 AÑOS', '5 - 14 AÑOS', '15 - 19 AÑOS',
       '20-39 AÑOS', '40-69 AÑOS', '>=70 AÑOS', 'n/a'], dtype=object)

array(['1', '2', '3', '4', '5', '6', '7', 'n/a'], dtype=object)

,rango_etario,id_rango_etario
0,< 1 AÑO,1
2,1 - 4 AÑOS,2
4,5 - 14 AÑOS,3
10,15 - 19 AÑOS,4
11,20-39 AÑOS,5
12,40-69 AÑOS,6
13,>=70 AÑOS,7
674,n/a,n/a
676,< 1 AÑO,2
678,1 - 4 AÑOS,3


Vemos que es una numeración estandar de enteros ascendentes. Hay casos en que está mal clasificado, por ejemplo, hay registros que indican que el rango etario 1-4 años tiene el id=2, y otros que indican que el mismo rango etario tiene el id=3. Como no parece haber un código muy elaborado, podemos establecer el siguiente criterio:
1. < 1 año
2. 1-4 años
3. 5-14 años
4. 15-19 años
5. 20-39 años
6. 40-69 años
7. $>=70 años$

Luego:-1 representa el 'n/a'.
Podemos crear un diccionario y usarlo para mapear el campo id_rango_etario.
Antes de eso, analizaremos los casos en que el id_rango_etario no sea nulo y el rango etario si lo sea.

In [755]:
df_pato[(df_pato['id_rango_etario'].notnull()) & (df_pato['rango_etario']).isnull()]

,id_saps,saps,fecha,patologia_desc,agrupacion_cie10,patologia_cod,id_rango_etario,consulta_cantidad,rango_etario,sexo


El resultado anterior nos muestra que no tenemos casos donde tengamos un id_rango_etario válido. Si fuera así, podríamos establecer el rango etario usando el id. Pero como no hay con id válidos, aplicamos la limpieza indicada anteriormente.

In [756]:
dic_rango_etario={
    '< 1 AÑO':1,
    '1 - 4 AÑOS':2, 
    '5 - 14 AÑOS':3, 
    '15 - 19 AÑOS':4,
    '20-39 AÑOS':5, 
    '40-69 AÑOS':6, 
    '>=70 AÑOS':7, 
    'n/a':-1
}

df_pato['id_rango_etario']=df_pato['rango_etario'].str.strip().map(dic_rango_etario)
df_pato['id_rango_etario'].unique()



array([ 1,  2,  3,  4,  5,  6,  7, -1])

En el resultado final vemos cómo ahora sólo tenemos esos 8 valores, a pesar de que le pedimos que nos de 10 resultados usando head(). Esto nos dice que ahora están todos homologados respecto del rango etario y su id.

In [757]:
df_pato.isnull().sum()

id_saps              100777
saps                   1054
fecha                     0
patologia_desc            0
agrupacion_cie10          0
patologia_cod             0
id_rango_etario           0
consulta_cantidad         0
rango_etario              0
sexo                   1148
dtype: int64

# Analizando El Campo Sexo

Nuevamente, empecemos revisando los datos únicos en ese campo.

In [758]:
df_pato['sexo'].unique()

array(['F', 'M', nan], dtype=object)

Una vez más, tenemos nan. Lo que haremos es mapear los resultados: 'F' pasará a ser 'Femenino','M' 'Masculino' y nan a 'n/a'.

In [759]:
df_pato['sexo']=(
    df_pato['sexo']
    .astype(str)
    .map({'F':'Femenino','M':'Masculino','nan':'n/a','Femenino':'Femenino','Masculino':'Masculino'})
)
display(df_pato['sexo'].unique())
df_pato.isnull().sum()

array(['Femenino', 'Masculino', 'n/a'], dtype=object)

id_saps              100777
saps                   1054
fecha                     0
patologia_desc            0
agrupacion_cie10          0
patologia_cod             0
id_rango_etario           0
consulta_cantidad         0
rango_etario              0
sexo                      0
dtype: int64

# Analizando los id de los SAPS y los nombres de los mismos

Empecemos viendo los valores únicos para los SAPS y los valores únicos para id_saps

In [760]:
display(df_pato['saps'].unique())
display(df_pato['id_saps'].unique())
display(len(df_pato['saps'].unique()))

array(['Dr. CALMANASH', 'DR. GUILLERMO RAWSON', 'SAPUCAY', 'CICHERO',
       'DR MANUEL A. GONZALEZ', 'DR. ALBERTO LIFSCHITZ',
       'DR. ANIBAL MALVIDO', 'DR. BENJAMIN SERRANO', 'DR. FLIER',
       'DR. JOSE DE LOS REYES VIDAL', 'DR. KORIMBLUM',
       'DR. MANUEL CASSUSO', 'DR. MAURICIO OPEN', 'DR. PEDRO BLUGERMAN',
       'DR. PIRCHI', 'DR. ROSSI CANDIA', 'Dr. SANTIAGO LORENZO',
       'DR. SEMPER', 'DR. SUSSINI', 'ESPERANZA', 'GDOR. ELIAS GALVAN',
       'GÜEMES', 'ITATI', 'MARCELINO VERA', 'PASO MARTINEZ',
       'PRIMERA JUNTA', 'PUJOL', 'QUINTA FERRE', 'RIO PARANA',
       'SAN MARCOS', 'SANTA MARGARITA', 'SANTA MARTA', 'VILLA CHIQUITA',
       'VILLA GARCIA', 'DR. CHERCOFF', 'SANTA CATALINA',
       'OPERATIVOS TERRITORIALES', nan], dtype=object)

array(['45', '47', '90', '76', '41', '88', nan, '49', '38', '48', '32',
       '75', '91', '58', '82', '42', '37', '36', '31', '54', '34', '86',
       '69', '62', '79', '46', '73', '85', '89', '71', '70', '51', '127',
       '99', '#REF!'], dtype=object)

38

Vemos que tenemos 'nan' para ambos y '#REF!' para los id. Busquemos los casos donde no tengamos ni id ni nombre del saps. Para estos casos, reemplazaremos 'nan' y '#REF!' por 'n/a' para indicar la ausencia de valores.

In [761]:
indices_saps=df_pato[(df_pato['id_saps'].isnull()) & (df_pato['saps'].isnull())].index
df_pato.iloc[indices_saps].head()

,id_saps,saps,fecha,patologia_desc,agrupacion_cie10,patologia_cod,id_rango_etario,consulta_cantidad,rango_etario,sexo
77276,NaN,NaN,2023-01-01,CONTROL DE SALUD: EXAMEN MEDICO GENERAL ADULTO,PERSONAS EN CONTACTO CON LOS SERVICIOS DE SALU...,Z000,-1,-1,n/a,n/a
77277,NaN,NaN,2023-01-01,CONTROL DE SALUD: EXAMEN MEDICO GENERAL ADULTO,PERSONAS EN CONTACTO CON LOS SERVICIOS DE SALU...,Z000,-1,-1,n/a,n/a
77278,NaN,NaN,2023-01-01,CONTROL DE SALUD: EXAMEN MEDICO GENERAL ADULTO,PERSONAS EN CONTACTO CON LOS SERVICIOS DE SALU...,Z000,-1,-1,n/a,n/a
77279,NaN,NaN,2023-01-01,CONTROL DE SALUD: EXAMEN MEDICO GENERAL ADULTO,PERSONAS EN CONTACTO CON LOS SERVICIOS DE SALU...,Z000,-1,-1,n/a,n/a
77280,NaN,NaN,2023-01-01,CONTROL DE SALUD: EXAMEN MEDICO GENERAL ADULTO,PERSONAS EN CONTACTO CON LOS SERVICIOS DE SALU...,Z000,-1,-1,n/a,n/a


In [762]:
#para los casos donde no haya id_saps ni saps (ni id, ni nombre del saps)
#rellenamos ambos campos con 'n/a' para indicar ausencia de datos
df_pato.iloc[indices_saps,0:2]='n/a'
df_pato.iloc[indices_saps].head()

,id_saps,saps,fecha,patologia_desc,agrupacion_cie10,patologia_cod,id_rango_etario,consulta_cantidad,rango_etario,sexo
77276,n/a,n/a,2023-01-01,CONTROL DE SALUD: EXAMEN MEDICO GENERAL ADULTO,PERSONAS EN CONTACTO CON LOS SERVICIOS DE SALU...,Z000,-1,-1,n/a,n/a
77277,n/a,n/a,2023-01-01,CONTROL DE SALUD: EXAMEN MEDICO GENERAL ADULTO,PERSONAS EN CONTACTO CON LOS SERVICIOS DE SALU...,Z000,-1,-1,n/a,n/a
77278,n/a,n/a,2023-01-01,CONTROL DE SALUD: EXAMEN MEDICO GENERAL ADULTO,PERSONAS EN CONTACTO CON LOS SERVICIOS DE SALU...,Z000,-1,-1,n/a,n/a
77279,n/a,n/a,2023-01-01,CONTROL DE SALUD: EXAMEN MEDICO GENERAL ADULTO,PERSONAS EN CONTACTO CON LOS SERVICIOS DE SALU...,Z000,-1,-1,n/a,n/a
77280,n/a,n/a,2023-01-01,CONTROL DE SALUD: EXAMEN MEDICO GENERAL ADULTO,PERSONAS EN CONTACTO CON LOS SERVICIOS DE SALU...,Z000,-1,-1,n/a,n/a


In [763]:
df_pato.isnull().sum()

id_saps              99723
saps                     0
fecha                    0
patologia_desc           0
agrupacion_cie10         0
patologia_cod            0
id_rango_etario          0
consulta_cantidad        0
rango_etario             0
sexo                     0
dtype: int64

Analicemos lo siguiente para la imputación de datos. La cuestión es que habrá SAPS que sí tienen número de id, y habrá otros que no. Analizando los datos para cada SAPS, hay algunos registros que tienen el id en entero y el mismo id en string. Esto se debe seguro a errores de ingreso de datos: algunas veces se colocó 1 y otras veces se colocó '1'. Entonces, tomaremos los valores único para el campo SAPS que nos dará los nombres de cada SAPS y algún valor 'n/a' de la imputación anterior.
Luego, usando esos nombres, filtraremos el dataframe. Para ese filtro obtenemos los valores únicos de id. Eso nos dará un arreglo con los valores enteros y de string (que es el mismo valor, pero con diferente tipo de dato) o nos dará nan.
Cuando obtengamos el unique para cada nombre, nos retornará un arreglo. Tomamos el primer valor de ese arreglo, y ese será el valor del id. Pero, antes de realizar eso, transformaremos los valores nan a -1 para indicar que no tenemos id para ese SAPS. Igual, posteriormente se le dará una clave subrogada al crear el esquema estrella.
Pero otro problema más es que tenemos un archivo csv con datos accesorios de los saps: dirección, barrio, teléfono de contacto, etc. Ese csv no tiene los id de los saps y los nombres de los saps difieren con el nombre que está registrado en el archivo de consultas. Entonces, como son pocos nombres, haremos que la tabla de consultas tenga el mismo nombre para los saps usando un diccionario para el mapeo.

In [764]:
#creamos el diccionario para unificar los nombres y realizar el proceso
dict_transformacion_saps={
"Dr. CALMANASH": "Dr. Saul Calmanash",
"Dr. Saul Calmanash":"Dr. Saul Calmanash",
"DR. GUILLERMO RAWSON": "Dr. Guillermo Rawson",
"Dr. Guillermo Rawson":"Dr. Guillermo Rawson",
"SANTA RITA":"Dr. Jose de los Reyes Vidal",
"Dr. Jose de los Reyes Vidal":"Dr. Jose de los Reyes Vidal",
"RAWSON":"Dr. Guillermo Rawson",
"Cichero": "Dr. Romilio Monzón",
"CICHERO":"Dr. Romilio Monzón",
"Dr. ROMILIO MONZÓN":"Dr. Romilio Monzón",
"Dr. Romilio Monzón":"Dr. Romilio Monzón",
"CAÑADA QUIROZ":"Dr. Marcelino Vera",
"Dr. Marcelino Vera":"Dr. Marcelino Vera",
"DR MANUEL A. GONZALEZ":"Dr. Manuel Gonzalez",
"Dr. Manuel Gonzalez":"Dr. Manuel Gonzalez",
"DR. ALBERTO LIFSCHITZ":"Dr. Lifschitz",
"Dr. Lifschitz":"Dr. Lifschitz",
"DR. ANIBAL MALVIDO":"Dr. Anibal Malvido",
"Dr. Anibal Malvido":"Dr. Anibal Malvido",
"DR. BENJAMIN SERRANO":"Dr. Benjamin Serrano",
"Dr. Benjamin Serrano":"Dr. Benjamin Serrano",
"DR. FLIER": "Dr. JC Benitez Suarez",
'LOMAS':"Dr. JC Benitez Suarez",
"Dr. JC Benitez Suarez":"Dr. JC Benitez Suarez",
"DR. JOSE DE LOS REYES VIDAL":"Dr. Jose de los Reyes Vidal",
"Dr. Jose de los Reyes Vidal":"Dr. Jose de los Reyes Vidal",
"DR. KORIMBLUM": "Dr. Korimblum",
"Dr. Korimblum":"Dr. Korimblum",
"Dr. Manuel Cassuso":"Dr. Manuel Cassuso",
"DR. MANUEL CASSUSO":"Dr. Manuel Cassuso",
"DR. MAURICIO OPEN": "Dr. Mauricio Open",
"Dr. Mauricio Open":"Dr. Mauricio Open",
"DR. PEDRO BLUGERMAN":"Dr. Pedro Blugerman",
"Dr. Pedro Blugerman":"Dr. Pedro Blugerman",
"DR. PIRCHI":"Dr. Oscar Pirchi",
"Dr. Oscar Pirchi":"Dr. Oscar Pirchi",
"DR. ROSSI CANDIA":"Dr. Miguel Rossi Candia",
"Dr. Miguel Rossi Candia":"Dr. Miguel Rossi Candia",
"Dr. SANTIAGO LORENZO":"Dr. Santiago Lorenzo",
"Dr. Santiago Lorenzo":"Dr. Santiago Lorenzo",
"DR. SEMPER":"Dr. Semper",
"Dr. Semper":"Dr. Semper",
"DR. SUSSINI":"Dr. Sussini",
"Dr. Sussini":"Dr. Sussini",
"ESPERANZA":"Esperanza",
"Esperanza":"Esperanza",
"GDOR. ELIAS GALVAN":"Dr. Elias Galvan",
"Dr. Elias Galvan":"Dr. Elias Galvan",
"GÜEMES":"Guemes",
"Guemes":"Guemes",
"ITATI":"Itati",
"Itati":"Itati",
"MARCELINO VERA":"Dr. Marcelino Vera",
"Dr. Marcelino Vera":"Dr. Marcelino Vera",
"PASO MARTINEZ":"Paso Martinez",
"Paso Martinez":"Paso Martinez",
"PRIMERA JUNTA":"Primera Junta",
"Primera Junta":"Primera Junta",
"PUJOL":"Pujol",
"Pujol":"Pujol",
"QUINTA FERRE":"Quinta Ferre",
"Quinta Ferre":"Quinta Ferre",
"RIO PARANA":"Rio Parana",
"Rio Parana":"Rio Parana",
"SAN MARCOS":"San Marcos",
"San Marcos":"San Marcos",
"SANTA MARGARITA":"Santa Margarita",
"Santa Margarita":"Santa Margarita",
"SANTA MARTA":"Santa Marta",
"Santa Marta":"Santa Marta",
"VILLA CHIQUITA":"Villa Chiquita",
"Villa Chiquita":"Villa Chiquita",
"VILLA GARCIA":"Villa Garcia",
"Villa Garcia":"Villa Garcia",
"DR. CHERCOFF":"Dr. Juan Chercoff",
"Dr. Juan Chercoff":"Dr. Juan Chercoff",
"SANTA CATALINA":"Santa Catalina",
"Santa Catalina":"Santa Catalina",
"OPERATIVOS TERRITORIALES":"OPERATIVO TERRITORIALES",
"n/a":"n/a",
"CENTRO DE DIST. DE VACUNAS":"Centro de distribucion de Vacunas",
"Centro de distribucion de Vacunas":"Centro de distribucion de Vacunas",
"SAPUCAY":"Sapucay",
"Sapucay":"Sapucay"
}

#aplico una homogeneidad en los nombres
df_pato['saps']=df_pato['saps'].replace(dict_transformacion_saps)

In [765]:
#obtengo los valores únicos de los nombres de los saps
saps=df_pato['saps'].unique()
#reviso los valores únicos de los id para cada nombre de saps
for nom in saps:
    display("Saps:{}-Ids:{}".format(nom,df_pato[df_pato['saps']==nom]['id_saps'].unique()))

"Saps:Dr. Saul Calmanash-Ids:['45' nan]"

"Saps:Dr. Guillermo Rawson-Ids:['47']"

"Saps:Sapucay-Ids:['90' nan]"

"Saps:Dr. Romilio Monzón-Ids:['76' nan]"

"Saps:Dr. Manuel Gonzalez-Ids:['41' nan]"

...

"Saps:Villa Garcia-Ids:['70' nan]"

"Saps:Dr. Juan Chercoff-Ids:['51' nan]"

"Saps:Santa Catalina-Ids:['127' nan]"

"Saps:OPERATIVO TERRITORIALES-Ids:['99' nan]"

"Saps:n/a-Ids:['n/a']"

Notamos que hay saps que tienen dos numeros (que son iguales, pero con diferente tipo de datos): uno es un entero y otro un string. Pero, ese nan al final nos está diciendo que hay registros con el nombre del SAPS pero sin su código. Por lo tanto, lo que haremos es reemplazar todos los registros con el mismo nombre de saps por el valor de su id correspondiente para que quede uno solo. Se reemplazará tanto el string como el nan, ya que ambos son valores de id para el mismo SAPS.

In [766]:
#obtengo los valores únicos de los nombres de los saps
saps=df_pato['saps'].unique()
#recorro cada valor
for nom in saps:
    #recupero el primer id del arreglo de ids
    if len(df_pato[df_pato['saps']==nom]['id_saps'].unique())>0:
        id=df_pato[df_pato['saps']==nom]['id_saps'].unique()[0]
    else:
        continue
    #ahora obtengo el índide de los registros con el nombre de saps que estamos analizando
    index_cambio=df_pato[df_pato['saps']==nom].index
    #si obtengo que para un determinado saps no hay índices, los evito por ahora
    if pd.isna(id) or id=="n/a":
        continue
     #con los índices anteriores, reemplazo por el id pertinente
    df_pato.iloc[index_cambio,0]=int(pd.to_numeric(id))

df_pato['id_saps'].unique()

array([45, 47, 90, 76, 41, 88, nan, 49, 38, 48, 32, 75, 91, 58, 82, 42,
       37, 36, 31, 54, 34, 86, 69, 62, 79, 46, 73, 85, 89, 71, 70, 51,
       127, 99, 'n/a'], dtype=object)

In [767]:
df_pato[df_pato['saps']=='Dr. Saul Calmanash']['id_saps'].unique()

array([45], dtype=object)

Tendremos algunos saps sin id establecidos. Veamos cuáles son

In [768]:
saps_sin_id=df_pato[(df_pato['id_saps'].isnull()) | (df_pato['id_saps']=="n/a")]['saps'].unique()
display(df_pato[(df_pato['id_saps'].isnull()) | (df_pato['id_saps']=="n/a")]['saps'].unique())
for saps in saps_sin_id:
    if pd.isna(saps) or saps=="n/a":
        continue
    print("SAPS: {}. IDS: {}".format(saps,df_pato[df_pato['saps']==saps]['id_saps'].unique()))


array(['Dr. Anibal Malvido', 'Dr. Korimblum', 'Dr. Semper', 'Guemes',
       'n/a'], dtype=object)

SAPS: Dr. Anibal Malvido. IDS: [nan]
SAPS: Dr. Korimblum. IDS: [nan]
SAPS: Dr. Semper. IDS: [nan]
SAPS: Guemes. IDS: [nan]


El resultado de arriba nos muestra que, si bien existen registros para el saps "Dr Anibal Malvido", no se le ha asignado ningún id. EL mismo criterio se aplica a los otros saps. Entonces, para que todos tengan un id, lo que hacemos es tomar el mayor id en el dataframe y, a partir de allí, darles id sucesivos a los saps que no tengan id establecido.

In [769]:
#filtro los datos para los casos en que no sean nulos y no tengan 'n/a' en el id
#obtengo el máximo y le sumo 1
nvo_id=df_pato[(df_pato['id_saps'].notna()) & (df_pato['id_saps']!='n/a')]['id_saps'].max()+1
#obtengo los saps sin id
saps_sin_id=df_pato[(df_pato['id_saps'].isnull()) | (df_pato['id_saps']=="n/a")]['saps'].unique()
#recorro cada saps
for saps in saps_sin_id:
    #recupero los índices donde se presentan esos saps
    index_saps_sin_id=df_pato[df_pato['saps']==saps].index

    #en caso contrario, les asigno el id arbitrario y aumento el valor en 1
    df_pato.iloc[index_saps_sin_id,0]=nvo_id
    nvo_id+=1
    


In [770]:
#muestra que los saps anteriores si obtuvieron un nuevo id
for saps in saps_sin_id:
    if pd.isna(saps) or saps=="n/a":
        continue
    print("SAPS: {}. IDS: {}".format(saps,df_pato[df_pato['saps']==saps]['id_saps'].unique()))


SAPS: Dr. Anibal Malvido. IDS: [128]
SAPS: Dr. Korimblum. IDS: [129]
SAPS: Dr. Semper. IDS: [130]
SAPS: Guemes. IDS: [131]


In [771]:
#cambio los id a enteros
#df_pato['ID_SAPS']=df_pato['ID_SAPS'].astype(int)
index_noid_nosaps=df_pato[(df_pato['id_saps'].isnull()) & (df_pato['saps'].isnull())].index
df_pato.iloc[index_noid_nosaps,0]=-1
df_pato.iloc[index_noid_nosaps,1]="n/a"
df_pato[df_pato['id_saps'].isnull()]


,id_saps,saps,fecha,patologia_desc,agrupacion_cie10,patologia_cod,id_rango_etario,consulta_cantidad,rango_etario,sexo


El último paso será darle los id permitentes al dataframe de saps para que queden igualados. El archivo con los datos de los saps no tiene el id para cada saps, así que le daremos el id que trae el conjunto de datos de las consultas por patología. De esa manera, se tendrá una homogeneidad para los id propios de cada saps.

In [772]:
#cargo los datos con la información de los saps(barrio, teléfono, etc)
df_saps=pd.read_sql("SELECT * FROM bronze.datosctes_saps", engine)
#Cuando se carga desde la base de datos, ocurre que los NULL se convierten en '#N/A' para los strings
#Y se convierten en None para los campos enteros
#Para facilitar el trabajo, convertiremos los #N/A es NaN
df_saps=df_saps.replace("#N/A",np.nan)
df_saps = df_saps.replace({None: np.nan})
#creo la nueva columna id_saps
df_saps.insert(0,"id_saps",[0 for i in np.arange(0,len(df_saps))])
#hago homogeneo los nombres de los saps
df_saps['saps']=df_saps['saps'].replace(dict_transformacion_saps)
nom_saps=df_pato['saps'].unique()
#recorro los nombres de los saps, analizando el dataframe con los datos de los saps
for saps in nom_saps:
    #si el saps que está en patologías figura en la lista de saps
    #obtengo el id del saps que está en la tabla de patologías
    #y se lo asigno al mismo saps en la lista de saps
    if len(df_saps[df_saps['saps']==saps])>0:
        id_saps=df_pato[df_pato['saps']==saps]['id_saps'].unique()[0]
        df_saps.iloc[df_saps[df_saps['saps']==saps].index,0]=id_saps
        #print(id_saps)
    
df_saps

,id_saps,saps,barrio,ubicacion,contacto_telefono,responsable,cargo
0,76,Dr. Romilio Monzón,Cichero,Guido Spano y Velez Sarfield,3794570325,Suarez Nancy,Director
1,128,Dr. Anibal Malvido,Pirayui,Suecia y Guillermo Ojeda,3794521654,Atoche Gennel Roland,Director
2,49,Dr. Benjamin Serrano,Parque Cadenas,Ruta 5 km.2 1/2,3794293513,Rezett Diego,Director
3,31,Dr. Elias Galvan,Galvan,Alverdi 2500 esq. Pizarro,3794815817,Lezcano Walter Ruben,Director
4,38,Dr. JC Benitez Suarez,Lomas,Mz. 1 Casa 1. Calles S. Mancini y Juan Pa.II,3794033767,Romero Monica,Director
5,47,Dr. Guillermo Rawson,San Geronimo,Centenario y Laprida,3794402928,Gomez Silvia,Director
6,48,Dr. Jose de los Reyes Vidal,Santa Rita,Perez ruedas 1989,3794783210,Saucedo Mario,Director
7,51,Dr. Juan Chercoff,Yecoha,Ruta 12 - Km 1040. Ramona Galarza s/n,3794694771,Obregon Walter,Director
8,129,Dr. Korimblum,NULL,Guemes y Berazategui,3794595798,Blanchard Julio,Director
9,88,Dr. Lifschitz,San Antonio,Horacio Quiroga esq. Roldan Belizario,3794765285,Mez Julio Jose,Director


In [773]:
#puede darse el caso que haya saps en la lista de saps que no figuren en la lista
#de consultas por patologías. Por ejemplo, en el conjunto de datos de las consultas
#no figura que se haya hecho alguna consulta a ese saps.
#Entonces, le asignamos un id de la misma manera fue asignado antes.
#Ahora, se podrán tomar los id de los saps desde el archivo con los datos de los saps, pero esto es sólo una 
#solución transitoria. Si cada saps tiene un id específico, cuando se tenga ese dato, deberá realizarse el
#reemplazo pertinente
max_id_saps=df_saps['id_saps'].max()+1
for i in df_saps[df_saps['id_saps']==0].index:
    df_saps.iloc[i,0]=max_id_saps
    max_id_saps+=1

df_saps

,id_saps,saps,barrio,ubicacion,contacto_telefono,responsable,cargo
0,76,Dr. Romilio Monzón,Cichero,Guido Spano y Velez Sarfield,3794570325,Suarez Nancy,Director
1,128,Dr. Anibal Malvido,Pirayui,Suecia y Guillermo Ojeda,3794521654,Atoche Gennel Roland,Director
2,49,Dr. Benjamin Serrano,Parque Cadenas,Ruta 5 km.2 1/2,3794293513,Rezett Diego,Director
3,31,Dr. Elias Galvan,Galvan,Alverdi 2500 esq. Pizarro,3794815817,Lezcano Walter Ruben,Director
4,38,Dr. JC Benitez Suarez,Lomas,Mz. 1 Casa 1. Calles S. Mancini y Juan Pa.II,3794033767,Romero Monica,Director
5,47,Dr. Guillermo Rawson,San Geronimo,Centenario y Laprida,3794402928,Gomez Silvia,Director
6,48,Dr. Jose de los Reyes Vidal,Santa Rita,Perez ruedas 1989,3794783210,Saucedo Mario,Director
7,51,Dr. Juan Chercoff,Yecoha,Ruta 12 - Km 1040. Ramona Galarza s/n,3794694771,Obregon Walter,Director
8,129,Dr. Korimblum,NULL,Guemes y Berazategui,3794595798,Blanchard Julio,Director
9,88,Dr. Lifschitz,San Antonio,Horacio Quiroga esq. Roldan Belizario,3794765285,Mez Julio Jose,Director


# Trabajando Con La Tabla De Inmunizaciones

La tabla de datos de las inmunizaciones servirá posteriormente para crear una tabla de hechos y dimensiones propias, que compartirá algunas tablas de hechos con las consultas por patología.
A continuación, se muestran los pasos para la limpieza de este conjunto de datos.

In [774]:
#cargo el dataframe
df_inmu=pd.read_sql("SELECT * FROM bronze.datosctes_inmunizacion", engine)
#Cuando se carga desde la base de datos, ocurre que los NULL se convierten en '#N/A' para los strings
#Y se convierten en None para los campos enteros
#Para facilitar el trabajo, convertiremos los #N/A es NaN
df_inmu=df_inmu.replace("#N/A",np.nan)
df_inmu = df_inmu.replace({None: np.nan})
df_inmu.info()
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27004 entries, 0 to 27003
Data columns (total 26 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   id_saps                      18862 non-null  object 
 1   saps                         26003 non-null  object 
 2   fecha                        26003 non-null  object 
 3   mes                          26999 non-null  object 
 4   anio                         26999 non-null  object 
 5   vacunas_tipo                 25984 non-null  object 
 6   vacunas_cantidad             25983 non-null  object 
 7   barrio_del_operativo         404 non-null    object 
 8   aviso_operativo_territorial  0 non-null      float64
 9   unnamed_9                    1 non-null      object 
 10  unnamed_10                   0 non-null      float64
 11  unnamed_11                   0 non-null      float64
 12  unnamed_12                   0 non-null      float64
 13  unnamed_13      

C:\Users\espin\AppData\Local\Temp\ipykernel_1520\2836967956.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_inmu = df_inmu.replace({None: np.nan})


Del resultado de arriba, notamos que tenemos muchos campos basura. Sólo nos quedaremos con los campos:  ID_SAPS, SAPS, fecha, vacunas_tipo,vacunas_cantidad.

In [775]:
#obtenemos sólo las columnas que queremos analizar
df_inmu=df_inmu[['id_saps','saps','fecha','vacunas_tipo','vacunas_cantidad']]
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
df_inmu.info()

Total Duplicados para las inmunizaciones: 1451
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27004 entries, 0 to 27003
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id_saps           18862 non-null  object
 1   saps              26003 non-null  object
 2   fecha             26003 non-null  object
 3   vacunas_tipo      25984 non-null  object
 4   vacunas_cantidad  25983 non-null  object
dtypes: object(5)
memory usage: 1.0+ MB


In [776]:
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
df_inmu[(df_inmu['id_saps'].isna())].head()

Total Duplicados para las inmunizaciones: 1451


,id_saps,saps,fecha,vacunas_tipo,vacunas_cantidad
44,NaN,DR. SEMPER,2020-01-01,HEPATITIS B,1
45,NaN,DR. SEMPER,2020-01-01,PENTAVALENTE,3
46,NaN,DR. SEMPER,2020-01-01,TRIPLE VIRAL,5
47,NaN,DR. SEMPER,2020-01-01,ANTI GRIPAL,4
48,NaN,DR. SEMPER,2020-01-01,HEPATITIS A,3


Podemos buscar los saps sin id y sin nombre de saps. De esa manera, los registros que estén en esa condición, no podremos encontrar una forma de saber a qué saps pertenece.

In [777]:
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
df_inmu[(df_inmu['id_saps'].notnull() & (df_inmu['saps'].isnull()))]

Total Duplicados para las inmunizaciones: 1451


,id_saps,saps,fecha,vacunas_tipo,vacunas_cantidad


Revisemos qué id de saps tenemos en la lista de saps y qué id de saps tenemos en los registros de inmunizaciones. De esa manera, sabremos qué id están en inmunizaciones y no en saps.

In [778]:
#transformamos en un conjunto los ids únicos de los saps
set_id_saps=set(df_saps['id_saps'].unique())
#hacemos lo mismo para las inmunizaciones, sólo que los nan y #REF! se
#reemplazan por -1
set_id_inmu=set(df_inmu['id_saps'].fillna(-1).replace({'#REF!':-1}).astype(int).unique())
#aplicamos la diferencia entre ambos conjuntos
print("Los id que están en inmunizaciones y no en los saps: {}".format(set_id_inmu.difference(set_id_saps)))
#print(set_id_inmu)
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
df_inmu.info()

Los id que están en inmunizaciones y no en los saps: {np.int64(99), np.int64(-1)}
Total Duplicados para las inmunizaciones: 1451
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27004 entries, 0 to 27003
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id_saps           18862 non-null  object
 1   saps              26003 non-null  object
 2   fecha             26003 non-null  object
 3   vacunas_tipo      25984 non-null  object
 4   vacunas_cantidad  25983 non-null  object
dtypes: object(5)
memory usage: 1.0+ MB


El resultado anterior nos muestra los id que figuran en el archivo de inmunizaciones, pero no en el archivo de saps. El -1 hace referencia a campos que no tienen id asignado. Observemos primero para los id 99.

In [779]:
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
display(df_inmu[df_inmu['id_saps']=='99'].head(1))

Total Duplicados para las inmunizaciones: 1451


,id_saps,saps,fecha,vacunas_tipo,vacunas_cantidad
3434,99,OPERATIVOS TERRITORIALES,01-07-2020,HEPATITIS B,1


El -1 indica que no hay datos del id. Esto es así porque aun no he transformado los nombres de los saps en el dataframe de las inmunizaciones para que se igualen a los que figuran en las consultas por patología y los saps. Voy a realizar esa transformación y ver cómo se modifican los id que están presentes en las inmunizaciones y que no están presentes en los saps.

In [780]:

#quitamos los espacios en blanco
df_inmu['saps']=df_inmu['saps'].str.strip()
#transformo los nombres para igualarlos con los del saps.
df_inmu['saps']=df_inmu['saps'].replace(dict_transformacion_saps)
#tranformamos los nombres de los saps para que sea igual a los nombres de los otros
#recorro todos los saps
for saps in df_saps['saps'].unique():
    #si el saps figura en la tabla de inmunizaciones
    if len(df_inmu[df_inmu['saps']==saps])>0:
        df_inmu.iloc[df_inmu[df_inmu['saps']==saps].index,0]=df_saps[df_saps['saps']==saps]['id_saps'].unique()[0]

print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
df_inmu[df_inmu['id_saps'].isnull()]['saps'].unique()


Total Duplicados para las inmunizaciones: 1466


array([nan], dtype=object)

Con el resultado anterior muestra que ya no hay ids nulos para los saps en inmunizaciones

In [781]:
df_inmu.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27004 entries, 0 to 27003
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id_saps           26003 non-null  object
 1   saps              26003 non-null  object
 2   fecha             26003 non-null  object
 3   vacunas_tipo      25984 non-null  object
 4   vacunas_cantidad  25983 non-null  object
dtypes: object(5)
memory usage: 1.0+ MB


In [782]:
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
df_inmu[(df_inmu['id_saps'].isnull()) & (df_inmu['saps'].notnull())]

Total Duplicados para las inmunizaciones: 1466


,id_saps,saps,fecha,vacunas_tipo,vacunas_cantidad


El resultado de arriba, nos dice que no tenemos nombres de saps sin nulos y que, a la vez, tengamos id_saps nulos.

In [783]:
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
df_inmu[(df_inmu['id_saps'].isnull()) & (df_inmu['saps'].notnull())].head()

Total Duplicados para las inmunizaciones: 1466


,id_saps,saps,fecha,vacunas_tipo,vacunas_cantidad


Ahora vemos que sí hay datos sin id_saps ni nombre de saps. Además, vemos que tenemos datos con los 5 campos sin datos. Estos registros no son útiles para nada. Una solución sería buscar los índices que cumplan esta condición y eliminarlos.

In [784]:
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
#revisamos una vez más, y notamos que ya no hay casos donde ambos: nombre e id de saps, 
#sean nulos a la vez.
df_inmu[(df_inmu['id_saps'].isnull()) & (df_inmu['saps'].isnull())].head()

Total Duplicados para las inmunizaciones: 1466


,id_saps,saps,fecha,vacunas_tipo,vacunas_cantidad
26003,NaN,NaN,NaN,NaN,NaN
26004,NaN,NaN,NaN,NaN,NaN
26005,NaN,NaN,NaN,NaN,NaN
26006,NaN,NaN,NaN,NaN,NaN
26007,NaN,NaN,NaN,NaN,NaN


Revisemos algunos casos más: 
1. Donde tenemos valores para las vacunas aplicadas, pero no sabemos su tipo.
2. Donde tenemos el tipo de vacuna, pero no su cantidad aplicada.
3. Donde tenemos cantidad de vacunas aplicadas, pero no tenemos sus saps.
4. Donde tenemos tipo de vacunas aplicadas, pero no tenemos su saps.
5. Donde no tengamos ni tipo de vacunas ni cantidad de vacunas.

In [785]:
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
#1.revisamos donde tenemos las cantidades, pero no los tipos de vacunas
df_inmu[(df_inmu['vacunas_cantidad'].notnull()) & (df_inmu['vacunas_tipo'].isnull())]

Total Duplicados para las inmunizaciones: 1466


,id_saps,saps,fecha,vacunas_tipo,vacunas_cantidad
19405,129,Dr. Korimblum,01/08/2023,NaN,3
20440,37,Dr. Sussini,01/11/2023,NaN,23


In [786]:
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
#2.revisamos los casos donde tenemos los tipos de vacunas, pero no las cantidades.
df_inmu[(df_inmu['vacunas_cantidad'].isnull()) & (df_inmu['vacunas_tipo'].notnull())]

Total Duplicados para las inmunizaciones: 1466


,id_saps,saps,fecha,vacunas_tipo,vacunas_cantidad
24,47,Dr. Guillermo Rawson,2020-01-01,SABIN ORAL,NaN
8443,99,OPERATIVO TERRITORIALES,01/06/2021,PENTAVALENTE,NaN
8469,82,Dr. Miguel Rossi Candia,01/07/2021,HEPATITIS B,NaN


In [787]:
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
#3.Donde tenemos la cantidad de vacunas, pero no los saps
df_inmu[(df_inmu['vacunas_cantidad'].notnull()) & (df_inmu['saps'].isnull())]

Total Duplicados para las inmunizaciones: 1466


,id_saps,saps,fecha,vacunas_tipo,vacunas_cantidad


In [788]:
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
#4.Donde tenemos los tipos de vacunas, pero no los saps
df_inmu[(df_inmu['vacunas_tipo'].notnull()) & (df_inmu['vacunas_tipo'].isnull())]

Total Duplicados para las inmunizaciones: 1466


,id_saps,saps,fecha,vacunas_tipo,vacunas_cantidad


In [789]:
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
#5.Donde no tengamos ni tipo de vacunas ni cantidad de vacunas.
df_inmu[(df_inmu['vacunas_tipo'].isnull()) & (df_inmu['vacunas_tipo'].isnull())].sort_values("fecha",ascending=True)

Total Duplicados para las inmunizaciones: 1466


,id_saps,saps,fecha,vacunas_tipo,vacunas_cantidad
2389,47,Dr. Guillermo Rawson,01-06-2020,NaN,NaN
17422,48,Dr. Jose de los Reyes Vidal,01/04/2023,NaN,NaN
19405,129,Dr. Korimblum,01/08/2023,NaN,3
19900,76,Dr. Romilio Monzón,01/10/2023,NaN,NaN
20440,37,Dr. Sussini,01/11/2023,NaN,23
...,...,...,...,...,...
26999,NaN,NaN,NaN,NaN,NaN
27000,NaN,NaN,NaN,NaN,NaN
27001,NaN,NaN,NaN,NaN,NaN
27002,NaN,NaN,NaN,NaN,NaN


De lo analizado anteriormente, vemos que se cumplen sólo los casos 1, 2 y 5. Podemos colocar los tipos de vacunas al valor "n/a" y las cantidades a -1 para indicar la ausencia de datos

In [790]:
#indices para los registros con cantidad de vacunas aplicadas, pero sin el tipo
indice_cantidad_sin_tipo=df_inmu[(df_inmu['vacunas_cantidad'].notnull()) & (df_inmu['vacunas_tipo'].isnull())].index
#el 3 indica que es la columna de índice 3 la que indica el tipo de vacuna
df_inmu.iloc[indice_cantidad_sin_tipo,3]="n/a"
df_inmu[(df_inmu['vacunas_cantidad'].notnull()) & (df_inmu['vacunas_tipo']=="n/a")]
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
df_inmu.info()

Total Duplicados para las inmunizaciones: 1466
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27004 entries, 0 to 27003
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id_saps           26003 non-null  object
 1   saps              26003 non-null  object
 2   fecha             26003 non-null  object
 3   vacunas_tipo      25986 non-null  object
 4   vacunas_cantidad  25983 non-null  object
dtypes: object(5)
memory usage: 1.0+ MB


In [791]:
#indices para los registros con cantidad de vacunas aplicadas, pero sin el tipo
indice_tipo_sin_cantidad=df_inmu[(df_inmu['vacunas_cantidad'].isnull()) & (df_inmu['vacunas_tipo'].notnull())].index
#el 4 indica que es la columna de índice 4, la cual representa la cantidad de vacunas
df_inmu.iloc[indice_tipo_sin_cantidad,4]=-1
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
df_inmu[(df_inmu['vacunas_cantidad']==-1) & (df_inmu['vacunas_tipo'].notnull())]


Total Duplicados para las inmunizaciones: 1466


,id_saps,saps,fecha,vacunas_tipo,vacunas_cantidad
24,47,Dr. Guillermo Rawson,2020-01-01,SABIN ORAL,-1
8443,99,OPERATIVO TERRITORIALES,01/06/2021,PENTAVALENTE,-1
8469,82,Dr. Miguel Rossi Candia,01/07/2021,HEPATITIS B,-1


Analicemos en quinto escenario

In [792]:
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
df_inmu[(df_inmu['vacunas_cantidad'].isnull()) & (df_inmu['vacunas_tipo'].isnull())]
df_inmu.info()

Total Duplicados para las inmunizaciones: 1466
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27004 entries, 0 to 27003
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id_saps           26003 non-null  object
 1   saps              26003 non-null  object
 2   fecha             26003 non-null  object
 3   vacunas_tipo      25986 non-null  object
 4   vacunas_cantidad  25986 non-null  object
dtypes: object(5)
memory usage: 1.0+ MB


Esto podríamos interpretarlo de dos maneras:
1. Amputamos los datos porque al no tener información sustancial, no serían útiles.
2. Los mantenemos porque a pesar de no tener el tipo ni la cantidad, nos dice que en esos días sí se aplicaron vacunas, aunque no sabemos cuántas y cuáles.

Por lo pronto, vamos a seguir con la misma idea: "n/a" para los tipos de vacunas, -1 para la cantidades, si es que se tienen datos NaN

In [793]:
index_ni_cantidad_ni_tipo=df_inmu[(df_inmu['vacunas_cantidad'].isnull()) & (df_inmu['vacunas_tipo'].isnull())].index
df_inmu.iloc[index_ni_cantidad_ni_tipo,3]='n/a'
df_inmu.iloc[index_ni_cantidad_ni_tipo,4]=-1
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
df_inmu.info()

Total Duplicados para las inmunizaciones: 1466
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27004 entries, 0 to 27003
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id_saps           26003 non-null  object
 1   saps              26003 non-null  object
 2   fecha             26003 non-null  object
 3   vacunas_tipo      27004 non-null  object
 4   vacunas_cantidad  27004 non-null  object
dtypes: object(5)
memory usage: 1.0+ MB


Y vemos que, finalmente, nos quedamos sin NaN en ninguna columna. Sólo falta darle a la fecha el formato pertinente, convertir los id en enteros al igual que las cantidades.
Un paso importante que no se hizo es revisar los valores que tiene el campo de cantidades de vacunas adminsitradas.

In [794]:
df_inmu['vacunas_cantidad'].unique()

array([-1, '1', '10', '5', '14', '4', '11', '30', '6', '3', '8', '32',
       '2', '7', '17', '12', '37', '31', '9', '33', '28', '22', '13',
       '15', '18', '16', '20', '39', '21', '23', '19', '60', '26', '51',
       '34', '40', '24', '29', '41', '25', '27', '53', '54', '62', '49',
       '35', '77', '158', '160', '110', '143', '149', '115', '50', '175',
       '157', '167', '118', '86', '63', '220', '215', '122', '109', '98',
       '69', '44', '45', '46', '43', '38', '95', '181', '196', '93',
       '165', '96', '176', '244', '67', '294', '70', '188', '64', '97',
       '349', '130', '178', '325', '104', '113', '144', '198', '108',
       '170', '298', '133', '258', '58', '36', '172', '120', '100', '163',
       '125', '66', '90', '156', '474', '131', '74', '65', '189', '48',
       '52', '56', '59', '55', '0', '225', '134', '71', '89', '83', '73',
       '42', '111', '123', '94', '47', '68', '150', '177', '132', '185',
       '72', '137', '161', '127', '107', '182', '84', '162',

Puede notarse que hay un valor '-' y '´1' que no son numéricos. Así que vamos a cambiarlo a -1 también para evitar problemas.

In [795]:
df_inmu.iloc[df_inmu[df_inmu['vacunas_cantidad']=='-'].index,4]=-1
df_inmu.iloc[df_inmu[df_inmu['vacunas_cantidad']=='´1'].index,4]=1
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))

Total Duplicados para las inmunizaciones: 1466


Ahora sí, cambiemos los tipos de datos.

In [796]:
df_inmu['fecha']=df_inmu['fecha'].str.replace("/","-")
df_inmu['fecha']=df_inmu['fecha'].str.strip()
df_inmu['fecha']=df_inmu['fecha'].apply(normalizar_fecha)
df_inmu['fecha']=pd.to_datetime(df_inmu['fecha'])
df_inmu['saps']=df_inmu['saps'].fillna("n/a")
df_inmu['id_saps']=df_inmu['id_saps'].fillna(-1).astype(int)
df_inmu['vacunas_cantidad']=df_inmu['vacunas_cantidad'].astype(int)
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
df_inmu['fecha'].unique()


Total Duplicados para las inmunizaciones: 1560


<DatetimeArray>
['2020-01-01 00:00:00', '2020-02-01 00:00:00', '2020-03-01 00:00:00',
 '2020-04-01 00:00:00', '2020-05-01 00:00:00', '2020-06-01 00:00:00',
 '2020-07-01 00:00:00', '1900-01-01 00:00:00', '2020-08-01 00:00:00',
 '2020-09-01 00:00:00', '2020-10-01 00:00:00', '2020-11-01 00:00:00',
 '2020-12-01 00:00:00', '2021-01-01 00:00:00', '2021-02-01 00:00:00',
 '2021-03-01 00:00:00', '2021-04-01 00:00:00', '2021-05-01 00:00:00',
 '2021-03-02 00:00:00', '2021-03-03 00:00:00', '2021-03-04 00:00:00',
 '2021-03-05 00:00:00', '2021-03-06 00:00:00', '2021-03-07 00:00:00',
 '2021-03-08 00:00:00', '2021-03-09 00:00:00', '2021-03-10 00:00:00',
 '2021-06-01 00:00:00', '2021-07-01 00:00:00', '2021-08-01 00:00:00',
 '2021-09-01 00:00:00', '2021-10-01 00:00:00', '2021-11-01 00:00:00',
 '2021-12-01 00:00:00', '2022-01-01 00:00:00', '2022-02-01 00:00:00',
 '2022-03-01 00:00:00', '2022-04-01 00:00:00', '2022-05-01 00:00:00',
 '2022-06-01 00:00:00', '2022-07-01 00:00:00', '2022-08-01 00:00:00',
 '20

Revisemos algunos datos más, como los valores en los tipos de vacunas.

In [797]:
df_inmu['vacunas_tipo']=df_inmu['vacunas_tipo'].str.strip()
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
df_inmu['vacunas_tipo'].unique()

Total Duplicados para las inmunizaciones: 1560


array(['n/a', 'ANTI GRIPAL', 'FIEBRE AMARILLA', 'DPT', 'DPT (a)', 'HPV',
       'HEPATITIS B', 'SALK', 'ROTAVIRUS', 'PENTAVALENTE',
       'NEUMOCOCICA CONJUGADA', 'SABIN ORAL', 'TRIPLE VIRAL',
       'HEPATITIS A', 'VARICELA', 'DOBLE BACTERIANA', 'MENINGOCOC.',
       'NEUMOCOCICA NO CONJUGADA', 'DOBLE VIRAL', 'MENVEO', 'BCG',
       'MENACTRA', 'TRIPLE BACTERIANA', 'NEUMO13', 'NEUMO 23', 'DTA',
       'GRIPE A', 'OTRAS', 'DOBLE ADULTO', 'NEUMO23', 'ANTIGRIPAL ADULTO',
       'ANTIGRIPAL PEDIATRICO', 'NEUMO 13', 'ANT. PEDIAT.',
       'ANTIGRIPAL PEDIA.', 'ANTI GRIPAL ADULTO', 'GRIPE', 'GRIPR A',
       'DT', 'DTO', 'ANTI AMARILLICA', 'ANTI GRIPAL PEDIATRICO',
       'BEXSERRO', 'PEDIAT', 'FLUXVIR', 'BEXSERO', 'O. SABIN', 'FLUXVIRG',
       'PREV 13', 'ANTIGRIPAL PEDITRICO', 'MENANTRA',
       'ANTIGRIPAL MAYORES', 'H', 'SA', 'OTROS', 'GRIPE +65',
       'PREVENAR 13', 'MAYOR 65', 'ANTIPEDIATRICO', 'ANT. PEDIAYTICO',
       'B', 'CAMPAÑA/22', 'GRIPAL', 'FLUVIA', 'A. CELULA', 'VSR',
  

Vemos que existen algunos que podrían ser iguales, pero con errores como nombres, espacios, etc:
- NEUMO13, NEUMO 13
- NUEMO23, NEUMO 23
- ANTIGRIPAL PEDIATRICO, ANTI GRIPAL PEDIATRICO, ANT. PEDIAYTICO
Podemos preparar los resultados con un diccionario para igualar los resultados. Además, podemos eliminar los espacios en blanco presentes.

In [798]:
#creo un diccionario donde cada vacuna será su clave y su valor
dict_vacunas = {v: v for v in df_inmu['vacunas_tipo'].unique()}

dict_vacunas



{'n/a': 'n/a',
 'ANTI GRIPAL': 'ANTI GRIPAL',
 'FIEBRE AMARILLA': 'FIEBRE AMARILLA',
 'DPT': 'DPT',
 'DPT (a)': 'DPT (a)',
 'HPV': 'HPV',
 'HEPATITIS B': 'HEPATITIS B',
 'SALK': 'SALK',
 'ROTAVIRUS': 'ROTAVIRUS',
 'PENTAVALENTE': 'PENTAVALENTE',
 'NEUMOCOCICA CONJUGADA': 'NEUMOCOCICA CONJUGADA',
 'SABIN ORAL': 'SABIN ORAL',
 'TRIPLE VIRAL': 'TRIPLE VIRAL',
 'HEPATITIS A': 'HEPATITIS A',
 'VARICELA': 'VARICELA',
 'DOBLE BACTERIANA': 'DOBLE BACTERIANA',
 'MENINGOCOC.': 'MENINGOCOC.',
 'NEUMOCOCICA NO CONJUGADA': 'NEUMOCOCICA NO CONJUGADA',
 'DOBLE VIRAL': 'DOBLE VIRAL',
 'MENVEO': 'MENVEO',
 'BCG': 'BCG',
 'MENACTRA': 'MENACTRA',
 'TRIPLE BACTERIANA': 'TRIPLE BACTERIANA',
 'NEUMO13': 'NEUMO13',
 'NEUMO 23': 'NEUMO 23',
 'DTA': 'DTA',
 'GRIPE A': 'GRIPE A',
 'OTRAS': 'OTRAS',
 'DOBLE ADULTO': 'DOBLE ADULTO',
 'NEUMO23': 'NEUMO23',
 'ANTIGRIPAL ADULTO': 'ANTIGRIPAL ADULTO',
 'ANTIGRIPAL PEDIATRICO': 'ANTIGRIPAL PEDIATRICO',
 'NEUMO 13': 'NEUMO 13',
 'ANT. PEDIAT.': 'ANT. PEDIAT.',
 'ANTIGR

Con los datos generados arriba, busco cuáles se repiten por errores de tipeo y, arreglando el diccionario, posteriormente lo usaré para homogeneizar los nombres de las vacunas.

In [799]:
#los errores en el diccionario por errores de tipeo, los arreglo
#{'ANTIGRIPAL PEDIA.':'ANTIGRIPAL PEDIA.'}
#{'ANTIGRIPAL PEDIA.':'ANTIGRIPAL PEDIATRICO'}
dict_vacunas['ANTIGRIPAL PEDIA.']='ANTIGRIPAL PEDIATRICO'
dict_vacunas['ANTI GRIPAL PEDIATRICO']='ANTIGRIPAL PEDIATRICO'
dict_vacunas['ANTIGRIPAL PEDITRICO']='ANTIGRIPAL PEDIATRICO'
dict_vacunas['ANT. PEDIAYTICO']='ANTIGRIPAL PEDIATRICO'
dict_vacunas['ANTIPEDIATRICO']='ANTIGRIPAL PEDIATRICO'
dict_vacunas['ANT. PEDIAT.']='ANTIGRIPAL PEDIATRICO'
dict_vacunas['ANTI GRIPAL ADULTO']='ANTIGRIPAL ADULTO'
dict_vacunas['GRIPR A']='GRIPE A'
dict_vacunas['ANTI AMARILLICA']='FIEBRE AMARILLA'
#dict_vacunas['ANTI AMARILLICA']='FIEBRE AMARILLA'

df_inmu['vacunas_tipo']=df_inmu['vacunas_tipo'].map(dict_vacunas)
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
df_inmu['vacunas_tipo'].unique()

Total Duplicados para las inmunizaciones: 1560


array(['n/a', 'ANTI GRIPAL', 'FIEBRE AMARILLA', 'DPT', 'DPT (a)', 'HPV',
       'HEPATITIS B', 'SALK', 'ROTAVIRUS', 'PENTAVALENTE',
       'NEUMOCOCICA CONJUGADA', 'SABIN ORAL', 'TRIPLE VIRAL',
       'HEPATITIS A', 'VARICELA', 'DOBLE BACTERIANA', 'MENINGOCOC.',
       'NEUMOCOCICA NO CONJUGADA', 'DOBLE VIRAL', 'MENVEO', 'BCG',
       'MENACTRA', 'TRIPLE BACTERIANA', 'NEUMO13', 'NEUMO 23', 'DTA',
       'GRIPE A', 'OTRAS', 'DOBLE ADULTO', 'NEUMO23', 'ANTIGRIPAL ADULTO',
       'ANTIGRIPAL PEDIATRICO', 'NEUMO 13', 'GRIPE', 'DT', 'DTO',
       'BEXSERRO', 'PEDIAT', 'FLUXVIR', 'BEXSERO', 'O. SABIN', 'FLUXVIRG',
       'PREV 13', 'MENANTRA', 'ANTIGRIPAL MAYORES', 'H', 'SA', 'OTROS',
       'GRIPE +65', 'PREVENAR 13', 'MAYOR 65', 'B', 'CAMPAÑA/22',
       'GRIPAL', 'FLUVIA', 'A. CELULA', 'VSR', 'SINCISIAL', 'HP'],
      dtype=object)

# Algunas consultas que podemos realizar

1. Crear una columna con los meses por año, agrupar las consultas por esa nueva columna, y obtener el total de consultas.

In [800]:
#obtengo el año y mes de cada fecha
#df_pato['y-m']=df_pato['fecha'].dt.to_period('M')
#df_pato['y-m'].head()

In [801]:
#agrupo los registros por los valores de la columna 'y-m' y aplico la agregación
#de suma. Reseteo los índices para que quede un nuevo dataframe
#df_pato.groupby('y-m')['consulta_cantidad'].sum().reset_index().head()

2. Cómo sumar meses, días o años a nuestras fechas

In [802]:
#suma 2 meses y 5 días a cada fecha
#El formato básico es: variable_fecha + pd.DateOffset(year=anios, months=meses, days=días)
#df_pato['fecha'] + pd.DateOffset(months=2,days=5)

3. Algunos filtros para las fechas

In [803]:
#filtrar las fechas sólo por el año y el mes
#df_pato[df_pato['fecha'].dt.strftime('%Y-%m')=='2020-06'].head()

In [804]:
#filtramos teniendo en cuenta sólo el mes
#df_pato[df_pato['fecha'].dt.month==6].tail()

# Control De Duplicados

In [805]:
df_inmu.info()
print(df_inmu.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27004 entries, 0 to 27003
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id_saps           27004 non-null  int64         
 1   saps              27004 non-null  object        
 2   fecha             27004 non-null  datetime64[ns]
 3   vacunas_tipo      27004 non-null  object        
 4   vacunas_cantidad  27004 non-null  int64         
dtypes: datetime64[ns](1), int64(2), object(2)
memory usage: 1.0+ MB
1560


In [806]:
#control de duplicados
#display(df_inmu.iloc[df_inmu.duplicated().index])
print("Total Duplicados para las patologias: {}".format(df_pato.duplicated().sum()))
print("Total Duplicados para los saps: {}".format(df_saps.duplicated().sum()))
print("Total Duplicados para las inmunizaciones: {}".format(df_inmu.duplicated().sum()))
#Una prueba del funcionamiento de drop_duplicates
#display(df_inmu[(df_inmu['ID_SAPS']==45) & (df_inmu['vacunas_tipo']=="VARICELA") ])
#display(df_inmu[(df_inmu['ID_SAPS']==45) & (df_inmu['vacunas_tipo']=="VARICELA") ].drop_duplicates())

Total Duplicados para las patologias: 2975
Total Duplicados para los saps: 0
Total Duplicados para las inmunizaciones: 1560


In [807]:
#Eliminación de duplicados
df_inmu=df_inmu.drop_duplicates()
df_pato=df_pato.drop_duplicates()
df_inmu.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25444 entries, 0 to 26003
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id_saps           25444 non-null  int64         
 1   saps              25444 non-null  object        
 2   fecha             25444 non-null  datetime64[ns]
 3   vacunas_tipo      25444 non-null  object        
 4   vacunas_cantidad  25444 non-null  int64         
dtypes: datetime64[ns](1), int64(2), object(2)
memory usage: 1.2+ MB


# Cargamos los datos a la capa de plata

In [808]:
table_name = "datosctes_consultas_patologia"
schema = "silver"   # ajustá si corresponde

with engine.begin() as conn:
    # 1. TRUNCATE (mucho más rápido que DELETE)
    conn.execute(
        text(f"TRUNCATE TABLE {schema}.{table_name}")
    )

    # 2. CARGA COMPLETA
    df_pato.to_sql(
        name=table_name,
        con=conn,
        schema=schema,
        if_exists="append",
        index=False,
        method="multi",
        chunksize=2000
    )


In [809]:
table_name = "datosctes_saps"
schema = "silver"   # ajustá si corresponde

with engine.begin() as conn:
    # 1. TRUNCATE (mucho más rápido que DELETE)
    conn.execute(
        text(f"TRUNCATE TABLE {schema}.{table_name}")
    )

    # 2. CARGA COMPLETA
    df_saps.to_sql(
        name=table_name,
        con=conn,
        schema=schema,
        if_exists="append",
        index=False,
        method="multi",
        chunksize=2000
    )

In [810]:
table_name = "datosctes_inmunizacion"
schema = "silver"   # ajustá si corresponde

with engine.begin() as conn:
    # 1. TRUNCATE (mucho más rápido que DELETE)
    conn.execute(
        text(f"TRUNCATE TABLE {schema}.{table_name}")
    )

    # 2. CARGA COMPLETA
    df_inmu.to_sql(
        name=table_name,
        con=conn,
        schema=schema,
        if_exists="append",
        index=False,
        method="multi",
        chunksize=2000
    )

In [811]:
#guardo el resultado final en un nuevo archivo csv para las consultas por patología

#df_pato.to_csv("consultas_por_patologia_limpio.csv",index=False)
#df_saps.to_csv("listado_saps_limpio.csv",index=False)
#df_inmu.to_csv("inmunizaciones_limpio.csv",index=False)